# PointDiT in motion: Kaggle evaluation

CPU only (Accelerator: None), Internet on. Attach the "PointDiT TUM RGB-D Subset" dataset and the output of the runner notebook version that wrote the predictions (it holds `predictions/<run name>/run.json`). Add the Kaggle secret `GITHUB_TOKEN` while the repo is private. No GPU, DINOv3, weights or `HF_TOKEN` are needed.

In [ ]:
PROJECT_REPO = "https://github.com/Sainava/Pointdit-In-Motion.git"
PROJECT_BRANCH = "feat/week1-data-metrics"
# None: use the single run.json found under /kaggle/input; set a path if several are attached.
PREDICTIONS_DIR = None
# Each worker needs ~3.5 GB of RAM; torch already uses every core for one frame.
WORKERS = 1

PROJECT_DIR = "/kaggle/working/Pointdit-In-Motion"
FORK_DIR = f"{PROJECT_DIR}/third_party/pointdit"
SEQUENCES_CONFIG = f"{PROJECT_DIR}/configs/sequences.yaml"

import base64
import os
import subprocess
import sys

from kaggle_secrets import UserSecretsClient


def github_token() -> str | None:
    try:
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        return None


# Only needed while the repo is private. Git reads the token from the environment, so it
# never appears in a command, in cell output, or in the clone's .git/config.
token = github_token()
if token:
    basic_auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    os.environ["GIT_CONFIG_COUNT"] = "1"
    os.environ["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
    os.environ["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {basic_auth}"
    del basic_auth
print("GitHub auth:", "token" if token else "none (public repo)")
del token

if os.path.isdir(f"{PROJECT_DIR}/.git"):
    !git -C {PROJECT_DIR} fetch -q origin {PROJECT_BRANCH}
    !git -C {PROJECT_DIR} checkout -q {PROJECT_BRANCH}
    !git -C {PROJECT_DIR} pull -q --ff-only origin {PROJECT_BRANCH}
else:
    !rm -rf {PROJECT_DIR}
    !git clone --recurse-submodules --branch {PROJECT_BRANCH} {PROJECT_REPO} {PROJECT_DIR}
if not os.path.isdir(f"{PROJECT_DIR}/.git"):
    raise RuntimeError(f"Cloning {PROJECT_REPO} failed; see the git output above.")
!git -C {PROJECT_DIR} submodule update --init --recursive
%cd {PROJECT_DIR}
!echo "project $(git -C {PROJECT_DIR} rev-parse HEAD)"
!echo "fork    $(git -C {FORK_DIR} rev-parse HEAD)"


def run(*cmd: object) -> None:
    """Run a project script, streaming its output, and fail the cell if it fails."""
    proc = subprocess.Popen([sys.executable, *map(str, cmd)], cwd=PROJECT_DIR, text=True,
                            env={**os.environ, "PYTHONUNBUFFERED": "1"},
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"{cmd[0]} exited with code {proc.returncode}; see the output above")

In [ ]:
!pip install -q "numpy>=1.26.0" "Pillow>=10.0.0" "matplotlib>=3.9.0" "PyYAML>=6.0" "torch>=2.4"

import torch

print("torch", torch.__version__, "| CPUs", os.cpu_count(), "| CUDA", torch.cuda.is_available())

In [ ]:
import csv
import json
from pathlib import Path

sys.path.insert(0, f"{PROJECT_DIR}/src")
from pim.data.tum import find_sequences

sequences = sorted(find_sequences("/kaggle/input"), key=lambda s: s.name)
for sequence in sequences:
    print(sequence)
data_roots = {s.parent for s in sequences}
if len(data_roots) != 1:
    raise RuntimeError(f"sequences are spread over several folders: {sorted(data_roots)}")
DATA_ROOT = data_roots.pop()

if PREDICTIONS_DIR is None:
    runs = sorted(Path("/kaggle/input").rglob("run.json"))
    if len(runs) != 1:
        raise RuntimeError(f"expected one run.json under /kaggle/input, found {[str(r) for r in runs]}; "
                           "set PREDICTIONS_DIR in the first cell")
    RUN_DIR = runs[0].parent
else:
    RUN_DIR = Path(PREDICTIONS_DIR)
run_record = json.loads((RUN_DIR / "run.json").read_text())
RUN_NAME = run_record["name"]
EVAL_DIR = Path("/kaggle/working/eval") / RUN_NAME
print(f"\npredictions {RUN_DIR} (run {RUN_NAME!r}, {run_record.get('start_time')} -> {run_record.get('end_time')})")
for name, seq in run_record["sequences"].items():
    print(f"  {name}: window {seq['window']}, {len(list((RUN_DIR / name).glob('*.npz')))} npz files")
print(f"results -> {EVAL_DIR}")

In [ ]:
run(f"{PROJECT_DIR}/scripts/evaluate.py", "--predictions", RUN_DIR, "--data-root", DATA_ROOT,
    "--sequences-config", SEQUENCES_CONFIG, "--out", EVAL_DIR, "--workers", WORKERS)

In [ ]:
CSV_PATH = EVAL_DIR / "per_frame.csv"
with open(CSV_PATH, newline="") as f:
    rows = list(csv.DictReader(f))
evaluated = list(dict.fromkeys(r["sequence"] for r in rows))

for name in evaluated:
    window = run_record["sequences"][name]["window"]
    middle = window["start"] + window["count"] // 2
    run(f"{PROJECT_DIR}/scripts/check_frame.py", "--predictions", RUN_DIR, "--data-root", DATA_ROOT,
        "--sequences-config", SEQUENCES_CONFIG, "--csv", CSV_PATH, "--sequence", name, "--frame", middle)
    print()

In [ ]:
WORST = 5

for name in evaluated:
    scored = [r for r in rows if r["sequence"] == name and r["rel_d"]]
    worst = sorted(scored, key=lambda r: float(r["rel_d"]), reverse=True)[:WORST]
    print(name, [(int(r["frame_index"]), round(float(r["rel_d"]), 2)) for r in worst])
    run(f"{PROJECT_DIR}/scripts/preview_predictions.py", "--predictions", RUN_DIR, "--data-root", DATA_ROOT,
        "--sequences-config", SEQUENCES_CONFIG, "--sequence", name,
        "--frames", *[r["frame_index"] for r in worst], "--out", EVAL_DIR / "worst_previews")

print("\nto download:")
for path in sorted(EVAL_DIR.rglob("*")):
    if path.is_file():
        print(f"  {path} ({path.stat().st_size / 1e3:.0f} kB)")